# Обработка данных волноводов LiNbO₃

COMSOL → контроль мод → полином n_eff(ω) выбранной степени → JSI и K.

**Выбранный способ:** аппроксимация Re(n_eff) полиномом выбранной степени по угловой частоте ω (рад/фс). По умолчанию используем степень 6; параметр POLYNOMIAL_DEGREE в разделе 4 позволяет выбрать одну степень от 1 до 6. Сравнение степеней сохранено в отдельном ноутбуке «LiNbO3 — расчёт волноводов — сравнения полиномов.ipynb». Этот выбор не заменяет проверку сходимости K.

Для получения исходных данных из терминала в папке проекта:
```powershell
python -m linbo3 --stage comsol --config run_config_comsol.json
```
Ноутбук читает готовые CSV/NPZ и не запускает COMSOL. Графики мод Шмидта и весов отключены.

## 1. Выбор расчёта

По умолчанию выбирается папка COMSOL-прогона с настройками из `run_config_comsol.json` и текущей версией модели. Для работы с другой сохранённой выгрузкой укажи `RESULT_DIR` вручную.


Проект можно переименовывать и переносить целиком. Запускай ноутбук из его корневой папки; после переноса перезапусти ядро. Все сохранённые ссылки на модель и данные относительные.


Хранение: один COMSOL-набор `results_comsol/current`; результаты этого ноутбука находятся в его профиле `analysis`. Новые настройки заменяют прежние данные метода.


In [ ]:
from pathlib import Path
from hashlib import file_digest
import json
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd()
PROJECT_DIR = cwd.resolve()  # ноутбук запускается из корня проекта
CONFIG_PATH = PROJECT_DIR / "run_config_comsol.json"
RESULT_DIR = None  # Например Path("results_comsol/current")

if not CONFIG_PATH.is_file():
    raise FileNotFoundError("Укажи папку LiNbO3 в PROJECT_DIR, в ней должен лежать run_config_comsol.json")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from linbo3.storage import set_profile
set_profile("main")


## 2. Загрузка готовых данных

`pair_status` показывает состояние пар «ширина × диапазон». В `raw_neff` попадают только полностью сохранённые пары. Мнимая часть и пропуски остаются в таблице. Переменная `result_dir` указывает на папку с CSV, NPZ и журналами COMSOL.

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8-sig"))
output_root = PROJECT_DIR / config["output_dir"]

if RESULT_DIR is not None:
    result_dir = PROJECT_DIR / Path(RESULT_DIR)
    if not (result_dir / "status.json").is_file():
        raise FileNotFoundError(result_dir / "status.json")
else:
    model_path = (PROJECT_DIR / config["model"]).resolve()
    with model_path.open("rb") as model_file:
        model_hash = file_digest(model_file, "sha256").hexdigest()
    candidates = []
    for manifest_file in [output_root / "current" / "config.json"]:
        if not manifest_file.is_file():
            continue
        manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
        same_config = all(manifest.get(key) == value for key, value in config.items()
                          if key not in {"model", "output_dir"})
        if (manifest.get("stage") == "comsol" and same_config
                and manifest.get("model_sha256") == model_hash
                and (manifest_file.parent / "status.json").is_file()):
            candidates.append(manifest_file)
    selected = max(candidates, key=lambda file: file.stat().st_mtime_ns, default=None)
    result_dir = selected.parent if selected else None

rows, frames = [], []
if result_dir is not None:
    status = json.loads((result_dir / "status.json").read_text(encoding="utf-8"))
    print(result_dir)
    pair_states = status.get("pairs")
    if pair_states is None:
        manifest = json.loads((result_dir / "config.json").read_text(encoding="utf-8"))
        pair_states = {f"{width:g}": {band: {"state": "complete"} for band in ("PUMP", "IS")}
                       for width in manifest["widths_um"]}
    for width, bands in pair_states.items():
        for band, pair in bands.items():
            state = pair["state"]
            csv_path = result_dir / f"w_{width}" / f"{band}_raw.csv"
            loaded = state == "complete" and csv_path.is_file()
            rows.append({"width_um": float(width), "band": band, "state": state,
                         "loaded": loaded, "error": pair.get("error")})
            if loaded:
                frames.append(pd.read_csv(csv_path).assign(width_um=float(width), band=band))

pair_status = pd.DataFrame(rows, columns=["width_um", "band", "state", "loaded", "error"])
raw_columns = ["width_um", "band", "wavelength_um", "outer_solnum", "solnum", "neff_real", "neff_imag"]
raw_neff = (pd.concat(frames, ignore_index=True).loc[:, raw_columns]
            if frames else pd.DataFrame(columns=raw_columns))

if result_dir is None:
    print("Подходящий прогон ещё не найден. Запусти COMSOL из терминала и повтори эту ячейку.")
else:
    print(f"Загружено пар {len(frames)} из {len(pair_status)}")
    display(pair_status)
    if not raw_neff.empty:
        display(raw_neff.head())

## 3. Re(n_eff), Im(n_eff) и контроль отбора — все ширины

Для каждой ширины показаны оба диапазона PUMP/IS. Серые точки — все решения COMSOL;
зелёные проходят `abs(Im(n_eff)) < IMAG_LIMIT` и проверку поля; цветные линии — подтверждённые ветви.
На Im(n_eff) отмечены ±IMAG_LIMIT, вертикально — опорные длины волн. Исходные точки сохранены без фильтрации.

Порог применяется строго **на каждой длине волны**, включая ручной выбор опорных решений.
`mode` — внутренний номер от 0 по возрастанию Re(n_eff) в опоре; `solnum` — номер решения COMSOL.
Внутренние номера здесь — идентификаторы. Физические подписи добавляются из классификации; в разделе 4 пара signal/idler выбирается по заданным пространственным меткам, даже если центральных IS-мод больше двух.
Если проверка не пройдена, причина будет в `pairs`, а точки останутся на графике.

In [ ]:
from linbo3.storage import analysis_root
from IPython.display import Image, display
from linbo3.saved_modes import inspect_modes

IMAG_LIMIT = 1e-6
MIN_OVERLAP = 0.9
MIN_MARGIN = 0.1
ANCHOR_SOLNUMS = None  # None: из манифеста; {}: полностью автоматически
# Пример явного выбора: {"0.7": {"PUMP": [3, 1], "IS": [2, 4]}}

tracking = dict(imag_limit=IMAG_LIMIT, min_overlap=MIN_OVERLAP, min_margin=MIN_MARGIN)
inspection = None
if result_dir is None:
    print("Сначала загрузи готовые данные в разделе 2.")
else:
    inspection = inspect_modes(result_dir, output_dir=analysis_root(result_dir) / "mode_diagnostics_poly6",
                               tracking=tracking, anchor_solnums=ANCHOR_SOLNUMS)
    # Вычисляем физические подписи до показа дисперсии; карты полей показываются в 3.1.
    import importlib
    import linbo3.mode_classification as mode_classification
    import linbo3.spatial_workflow as spatial_workflow
    importlib.reload(mode_classification)
    importlib.reload(spatial_workflow)
    classification_report = mode_classification.classify_run(
        result_dir, imag_limit=IMAG_LIMIT, polarization_threshold=.9,
        branches=inspection.branches,
    )
    spatial_workflow.plot_raw_spatial_labels(result_dir, inspection, classification_report.table)
    for width in sorted(inspection.pairs["width_um"].unique()):
        display(Image(filename=str(inspection.output_dir / f"w_{width:g}" / "neff_raw.png")))

## 3.1. Пространственная классификация при 1550 нм (IS) и 775 нм (PUMP)

Для каждой ширины: сверху — все допустимые центральные IS-моды, снизу — PUMP-моды. Классификация читает исходные комплексные Ex/Ey/Ez и работает даже при отказе прослеживания в разделе 3 (например, когда IS-мод больше двух).

В этой модели x лежит вдоль плёнки, y направлена поперёк плёнки, распространение — по z. **TE ≈ Ex, TM ≈ Ey**; это сокращённые обозначения quasi-TE/quasi-TM. Поляризация определяется долей ∫|Ex|² dA / ∫(|Ex|²+|Ey|²) dA в трапециевидной области ядра. Это доля квадрата электрического поля, не доля потока мощности.

**Номер = число боковых пиков − 1:** 1 пик → 0, 2 → 1, 3 → 2. Считаются пики профиля доминирующей поперечной компоненты, интегрированной по высоте волновода. На вставке видны этот профиль и отмеченные максимумы. Одноиндексная метка описывает боковой порядок; при нескольких вертикальных семействах её недостаточно.

Основная карта показывает **|E| = √(|Ex|²+|Ey|²+|Ez|²)**, нормированное отдельно для каждой моды. FIELD_VIEW='dominant' переключает её на |Ex| либо |Ey|. Нормировка нужна для сравнения формы, собственные решения не имеют общей абсолютной амплитуды. Сглаживание цвета применяется только при отображении; пики считаются по реальным сохранённым отсчётам.

Если доминирование поляризации меньше 90%, подпись **Hybrid** показывает смешанные семейства; первым указано семейство доминирующей компоненты. Знак **?** означает нестабильное число пиков, недостаточное разрешение или повторяющуюся пространственную метку. Такие случаи требуют проверки; алгоритм не назначает чистый TE/TM принудительно. Порог пиков и поляризации доступен ниже. Фильтр |Im(n_eff)| < IMAG_LIMIT сохраняется; ни отсутствие моды в выгрузке, ни интерполяция рисунка не доказывают отсутствие физической моды.

Метки **solnum**, **mode**, PUMP-ранг и **TE/TM-порядок** — разные величины. Эта ячейка добавляет классификацию и соответствия в classification.csv; последующие разделы используют эти пространственные метки. Пара signal/idler явно задаётся SIGNAL_MODE_LABEL и IDLER_MODE_LABEL. Ширины с более чем двумя допустимыми IS-модами исключаются из последующих расчётов JSI/K.

Основания: [COMSOL: анализ и идентификация мод](https://www.comsol.com/blogs/mode-analysis-for-electromagnetic-waveguides-in-comsol/), [пример quasi-TE/TM и одно-/двухлепестковых профилей](https://www.comsol.com/paper/download/103573/hutter_paper.pdf).


In [ ]:
import importlib
import linbo3.mode_classification as mode_classification

importlib.reload(mode_classification)

# TE/TM: оси этой модели, TE преимущественно Ex, TM преимущественно Ey.
CLASSIFICATION_POLARIZATION_THRESHOLD = 0.9
CLASSIFICATION_PEAK_PROMINENCE = 0.08
SIGNAL_MODE_LABEL = "TM0"  # Явная пространственная мода signal
IDLER_MODE_LABEL = "TE0"   # Явная пространственная мода idler
FIELD_VIEW = "total"  # "total": |E|; "dominant": модуль основной Ex/Ey-компоненты

classification_report = globals().get("classification_report")
if result_dir is None:
    print("Сначала загрузи готовые данные в разделе 2.")
else:
    settings_path = classification_report.output_dir / "settings.json" if classification_report is not None else None
    cached_settings = json.loads(settings_path.read_text(encoding="utf-8")) if settings_path is not None and settings_path.is_file() else {}
    classification_current = (classification_report is not None and classification_report.output_dir.parent == result_dir
        and cached_settings.get("polarization_threshold") == CLASSIFICATION_POLARIZATION_THRESHOLD
        and cached_settings.get("peak_prominence") == CLASSIFICATION_PEAK_PROMINENCE
        and cached_settings.get("imag_limit") == IMAG_LIMIT and cached_settings.get("field_view") == FIELD_VIEW)
    if not classification_current:
        classification_report = mode_classification.classify_run(
        result_dir, imag_limit=IMAG_LIMIT,
        polarization_threshold=CLASSIFICATION_POLARIZATION_THRESHOLD,
        peak_prominence=CLASSIFICATION_PEAK_PROMINENCE, field_view=FIELD_VIEW,
        branches=inspection.branches if inspection is not None else None,
    )
    for figure in classification_report.figures:
        display(Image(filename=str(figure)))
    if not classification_report.table.empty:
        review_count = int(classification_report.table["status"].eq("review").sum())
        if review_count:
            print(f"Метки Hybrid / ?: {review_count}. Подробности: {classification_report.output_dir / 'classification.csv'}")


## 4. Выбор степени полинома и производные по ω

Задай POLYNOMIAL_DEGREE (целое число 1–6) в следующей ячейке. По умолчанию выбрана степень 6. Аппроксимируется Re(n_eff) по ω; эта же выполненная аппроксимация определяет степень для JSI и K. После изменения степени повтори разделы 4–6. Сравнение степеней остаётся в отдельном ноутбуке.

In [ ]:
import importlib
import linbo3.spatial_workflow as spatial_workflow
importlib.reload(spatial_workflow)

POLYNOMIAL_DEGREE = 6
spatial_study = None
dispersion_study = None
if classification_report is None:
    print("Сначала выполни пространственную классификацию в разделе 3.1.")
else:
    spatial_study = spatial_workflow.prepare_spatial_study(
        result_dir, classification_report.table, degree=POLYNOMIAL_DEGREE, tracking=tracking,
        signal_label=SIGNAL_MODE_LABEL, idler_label=IDLER_MODE_LABEL,
    )
    dispersion_study = spatial_study.study
    spatial_workflow.plot_spatial_dispersion(spatial_study, classification_report.output_dir / "dispersion")


## 5. JSI и K по пространственным меткам

Строки — ширины, столбцы — пространственные PUMP-моды. В заголовках указаны физические метки; Hybrid выделен красным. Signal/idler выбраны явно в разделе 3.1. Результаты пишутся отдельно в spatial_poly<степень>_omega_2000fs. После перехода на физические метки первый запуск требует RUN_STUDY=True. Численные кэши с прежней нумерацией не смешиваются с новой схемой.


In [ ]:
import importlib
import linbo3.spatial_workflow as spatial_workflow
importlib.reload(spatial_workflow)

RUN_STUDY = False
STUDY_POINTS = 1501
study_dir = None
comparison_ready = False
if spatial_study is None:
    print("Сначала выполни разделы 2–4.")
else:
    study_dir = spatial_workflow.spatial_directory(result_dir, STUDY_POINTS, spatial_study.study.degrees[0])
    if RUN_STUDY:
        study_dir = spatial_workflow.run_spatial_study(result_dir, spatial_study, STUDY_POINTS)
    comparison_ready = spatial_workflow.spatial_saved_is_current(result_dir, spatial_study, STUDY_POINTS)
    if not comparison_ready:
        print("Нет актуальных JSI/K с пространственными метками. Установи RUN_STUDY = True.")
    else:
        # Только перерисовка готовых данных: общий заголовок ширины для каждого ряда.
        if not RUN_STUDY:
            spatial_workflow.plot_spatial_jsi(study_dir, spatial_study, run_dir=result_dir)
        display(Image(filename=str(study_dir / "figures/jsi_overview.png")))


## 5.1. Нормированные спектры signal/idler и FWHM

Из полной сохранённой JSI (не из уменьшенной картинки) считаются маргинальные спектры. Строки JSI соответствуют idler, столбцы — signal:

- S_s(ω_s) = Σ_i JSI(ω_i,ω_s) Δω_i;
- S_i(ω_i) = Σ_s JSI(ω_i,ω_s) Δω_s.

Используется та же равномерная частотная сетка и прямоугольная сумма, что в основном расчёте. Для вакуумной длины волны λ в нм: **S_λ(λ) = S_ω(2πc/λ) · 2πc/λ²**. Скорость света берётся из проекта, единицы согласованы: c в нм/фс. Длина волны упорядочивается по возрастанию.

Каждый signal/idler-спектр нормируется на свой максимум **после** преобразования координаты. FWHM рассчитывается отдельно по ω и по λ; она не получается простым пересчётом частотной ширины. Уровень половины максимума ищется вокруг главного максимума, пересечения интерполируются линейно. Для нескольких несвязных областей выше половины максимума подписывается «главн. пик». Если граница лежит за окном, вместо недостоверной ширины пишется «вне окна».

Ниже выводятся ровно две общие фигуры: по ω [рад/фс] и по λ [нм]. Ряды — ширины, столбцы — пространственные моды накачки; FWHM signal и idler указаны в легендах панелей. Гибридные заголовки выделены красным. Нового расчёта COMSOL или JSI/K не требуется. Численные FWHM и статусы сохраняются в marginal_fwhm.csv.


In [ ]:
import importlib
import linbo3.marginal_spectra as marginal_spectra
from linbo3.spatial_workflow import spatial_directory

importlib.reload(marginal_spectra)
SPECTRA_DPI = 150

if globals().get("spatial_study") is None or globals().get("result_dir") is None:
    print("Сначала выполни разделы 2–5.")
else:
    spectra_points = globals().get("STUDY_POINTS", 1501)
    spectra_dir = spatial_directory(result_dir, spectra_points, spatial_study.study.degrees[0])
    try:
        spectra_figures = marginal_spectra.plot_marginal_spectra(
            result_dir, spatial_study, spectra_dir, points=spectra_points, dpi=SPECTRA_DPI,
        )
    except (FileNotFoundError, ValueError) as error:
        print(error)
    else:
        for figure in spectra_figures:
            display(Image(filename=str(figure)))


## 5.2. Гибридизация накачки в спектральном интервале JSI

Суммируем **антидиагонали** полной JSI: каждая соответствует постоянной сумме $\omega_p=\omega_s+\omega_i$. Используем веса исходной равномерной сетки по $\omega$; ширины отдельных маргинальных спектров здесь не складываются.

`SPECTRAL_FRACTION = 0.99` задаёт центральный интервал с 99% вклада: по 0.5% в каждом хвосте. Границы — дискретные квантили, включённые отсчёты содержат не меньше заданной доли. Для длин волн порядок обращается: $\lambda_{min}=2\pi c/\omega_{max}$, $\lambda_{max}=2\pi c/\omega_{min}$.

Внутри интервала проверяем сохранённые комплексные поля **той же ветви накачки**, отслеженной по перекрытию от классифицированной центральной моды. Критерий: меньшая доля поперечной поляризации в ядре **строго больше** `MIXING_LIMIT = 0.10`. Параметры длительности импульса и спектральной сетки читаются из соответствующего рассчитанного случая JSI.

Число проверенных точек сохраняется явно: `single_sample` означает одну точку; `no_samples` — отсутствие точек; `outside_comsol_range` — неполное покрытие диапазоном COMSOL; `field_error` — ошибку проверки. Отсутствие обнаруженной гибридизации относится только к проверенным точкам, а не ко всему непрерывному интервалу. Новая JSI или более мелкая сетка полей учитываются после повторного запуска этой ячейки.

CSV: `pump_hybridization_cases.csv` — границы и итог по случаям, `pump_hybridization_samples.csv` — проверенные решения и доли поляризаций. В разделе 6 оранжевые ромбы меньше красных: красные обозначают прежнюю проверку в центре, оранжевые — накачку в этом интервале. При совпадении видны оба. Расчёт COMSOL, JSI и K не запускается.


In [ ]:
import importlib
import linbo3.pump_hybridization as pump_hybridization
from linbo3.spatial_workflow import spatial_directory
importlib.reload(pump_hybridization)

SPECTRAL_FRACTION = 0.99  # доля вклада полной JSI, по равным хвостам
MIXING_LIMIT = 0.10      # гибридизация: примесь строго больше 10%
hybridization_audit = None

if globals().get("spatial_study") is None or globals().get("result_dir") is None:
    print("Сначала выполни разделы 2–5.")
else:
    audit_points = globals().get("STUDY_POINTS", 1501)
    audit_dir = spatial_directory(result_dir, audit_points, spatial_study.study.degrees[0])
    try:
        hybridization_audit = pump_hybridization.audit_pump_hybridization(
            result_dir, spatial_study, audit_dir, points=audit_points,
            fraction=SPECTRAL_FRACTION, mixing_limit=MIXING_LIMIT)
    except (FileNotFoundError, ValueError, OSError) as error:
        print(error)
    else:
        cases = hybridization_audit.cases
        print(f"Проверено случаев: {len(cases)}. Гибридизация накачки обнаружена: "
              f"{int(cases['pump_interval_hybrid'].sum())}.")
        print("Проверенных точек COMSOL в одном случае: "
              f"{cases['checked_points'].min()}–{cases['checked_points'].max()}.")
        if cases['checked_points'].eq(1).any():
            print("Есть случаи с единственной точкой: это проверка только в этой точке, не всего интервала.")
        if cases['status'].isin(['no_samples', 'field_error', 'outside_comsol_range']).any():
            print("Есть случаи с неполной проверкой. Подробности: pump_hybridization_cases.csv.")
        print("Границы и проверенные поля сохранены в pump_hybridization_*.csv. Далее выполни раздел 6.")


## 6. Число Шмидта от ширины — все пространственные моды на одном графике

Каждая линия — одна пространственная мода накачки. Ширины с более чем двумя допустимыми IS-модами в центральной точке исключаются. Гибридные точки (примесь >10%) — красные ромбы. Старые численные результаты можно использовать только для текущего допустимого поднабора при совпадении исходных данных, пары, сетки и меток.


In [ ]:
import importlib
import linbo3.spatial_workflow as spatial_workflow
importlib.reload(spatial_workflow)

# Размер всей фигуры в дюймах: (ширина, высота). Их отношение задаёт пропорции.
K_FIGSIZE = (6, 4)
K_DPI =100  # Разрешение отображения и сохранённого PNG.

if globals().get("spatial_study") is None:
    print("Сначала выполни раздел 4.")
else:
    k_dir = spatial_workflow.spatial_directory(result_dir, globals().get("STUDY_POINTS", 1501), spatial_study.study.degrees[0])
    if not spatial_workflow.spatial_saved_is_current(result_dir, spatial_study, globals().get("STUDY_POINTS", 1501), allow_cached_subset=True):
        print("Сначала выполни расчёт JSI/K с пространственными метками в разделе 5.")
    else:
        if globals().get("hybridization_audit") is None:
            print("Для оранжевых отметок сначала выполни раздел 5.2.")
        else:
            try:
                spatial_workflow.plot_spatial_k(
                    k_dir, spatial=spatial_study, figsize=K_FIGSIZE, dpi=K_DPI,
                    hybridization=hybridization_audit)
            except ValueError as error:
                print(error)


## 7. JSI и число Шмидта для разных длительностей накачки

Серия: **250, 500, 750, 1000, 1250, 1500, 1750, 2000 фс**. Пространственные ветви, полином выбранной степени, длина взаимодействия и определение длительности импульса остаются такими же, как в разделе 5.

`PULSE_WINDOWS` — редактируемая таблица полуширин окна **±DW в рад/фс** для осей signal/idler. Начальные значения заданы линейно по 1/t между DW(250)=0.015 и DW(2000)=0.0025: `DW(t)=0.0025+(0.015-0.0025)*(1/t-1/2000)/(1/250-1/2000)`.

`PULSE_POINTS` задаёт число точек каждой оси. При большем DW и том же числе точек шаг увеличивается: при необходимости проверяй сходимость K увеличением `PULSE_POINTS`. Отсечение JSI на границах сопровождается предупреждением по `border_fraction`; это отдельная проверка от сходимости по шагу. Экстраполяция за пределы рассчитанных диапазонов COMSOL не допускается.

`RUN_PULSE_SERIES=True` рассчитывает только отсутствующие случаи и использует кэш готовых. После прерывания запуск можно повторить; законченные JSI сохраняются по одной. `False` загружает только готовую серию. Кэш разделён по длительности, DW, размеру сетки и выбранной аппроксимации. Готовые 2000 фс при ±0.0025 используются из раздела 5.

Для каждой длительности проверяем гибридизацию в её собственном интервале 99% JSI, с теми же параметрами доли и примеси, что в 5.2. В этой ячейке автоматически показываются **только графики K от ширины**, с красными и маленькими оранжевыми ромбами. Полные JSI и сводный CSV сохраняются; изображения JSI и спектров здесь не строятся и не добавляются в вывод ноутбука.

Дополнительно строится общая фигура K(t): для каждой ширины свой график, 3 столбца × 2 ряда; все пространственные моды и оба типа отметок гибридизации сохранены. Размер задаётся PULSE_DURATION_FIGSIZE.


In [ ]:
import importlib
import linbo3.cached_spectra as cached_spectra
import linbo3.spatial_workflow as spatial_workflow
import linbo3.marginal_spectra as marginal_spectra
import linbo3.pump_hybridization as pump_hybridization
import linbo3.pulse_series as pulse_series

# Подхватываем обновления модулей без перезапуска ядра.
for module in (cached_spectra, spatial_workflow, marginal_spectra, pump_hybridization, pulse_series):
    importlib.reload(module)

RUN_PULSE_SERIES = True
PULSE_POINTS = 2000
PULSE_WINDOWS = {
    250: 0.015,
    500: 0.007857142857142856,
    750: 0.005476190476190476,
    1000: 0.004285714285714285,
    1250: 0.0035714285714285718,
    1500: 0.003095238095238095,
    1750: 0.0027551020408163266,
    2000: 0.0025,
}
PULSE_SPECTRAL_FRACTION = globals().get("SPECTRAL_FRACTION", 0.99)
PULSE_MIXING_LIMIT = globals().get("MIXING_LIMIT", 0.10)
PULSE_K_FIGSIZE = (24, 11)  # размер всей фигуры: 4 столбца × 2 ряда
PULSE_K_DPI = 70
PULSE_DURATION_FIGSIZE = (18, 10)  # общая фигура: 3 столбца × 2 ряда
pulse_results = None

if globals().get("spatial_study") is None or globals().get("result_dir") is None:
    print("Сначала выполни разделы 2–4.")
else:
    try:
        pulse_results = pulse_series.run_pulse_series(
            result_dir, spatial_study, PULSE_WINDOWS, points=PULSE_POINTS,
            fraction=PULSE_SPECTRAL_FRACTION, mixing_limit=PULSE_MIXING_LIMIT,
            calculate=RUN_PULSE_SERIES)
    except (FileNotFoundError, ValueError, OSError) as error:
        print(error)
    else:
        pulse_series.plot_pulse_k_grid(
            result_dir, spatial_study, PULSE_WINDOWS,
            points=PULSE_POINTS, figsize=PULSE_K_FIGSIZE, dpi=PULSE_K_DPI)
        pulse_series.plot_k_duration_by_width(
            result_dir, spatial_study, PULSE_WINDOWS,
            points=PULSE_POINTS, figsize=PULSE_DURATION_FIGSIZE, dpi=PULSE_K_DPI)


## 7.1. Показать готовые JSI и спектры по запросу: пример 250 фс

Ячейка **только читает сохранённые матрицы** и перерисовывает изображения; расчёт JSI/K и SVD не запускается. Измени `SHOW_DURATION_FS` на рассчитанную длительность из `PULSE_WINDOWS`. Переключатели отдельно включают общую JSI и две общие картинки маргинальных спектров (по ω и по λ с якобианом, с FWHM). Ряды соответствуют ширинам, столбцы — пространственным модам накачки.

В примере `DISPLAY_PULSE_FIGURES=True`: выводится только выбранный случай 250 фс. Установи `False`, чтобы отключить и этот вывод при массовом выполнении ноутбука. Если готовых данных нет, она сообщит об этом, не начиная расчёт.


In [ ]:
from IPython.display import Image, display
import importlib
import linbo3.pulse_series as pulse_series
importlib.reload(pulse_series)

DISPLAY_PULSE_FIGURES = True  # True: показать выбранный случай из кэша
SHOW_DURATION_FS = 250
SHOW_JSI = True
SHOW_MARGINAL_SPECTRA = True
PULSE_FIGURES_DPI = 150

if not DISPLAY_PULSE_FIGURES:
    print("Для вывода готового случая 250 фс установи DISPLAY_PULSE_FIGURES=True.")
elif globals().get("spatial_study") is None or globals().get("PULSE_WINDOWS") is None:
    print("Сначала выполни разделы 2–4 и настройку серии в разделе 7.")
elif SHOW_DURATION_FS not in PULSE_WINDOWS:
    print("Добавь длительность и DW в PULSE_WINDOWS и рассчитай её в разделе 7.")
else:
    try:
        selected_figures = pulse_series.render_pulse_figures(
            result_dir, spatial_study, SHOW_DURATION_FS, PULSE_WINDOWS[SHOW_DURATION_FS],
            points=PULSE_POINTS, jsi=SHOW_JSI, spectra=SHOW_MARGINAL_SPECTRA,
            dpi=PULSE_FIGURES_DPI)
    except (FileNotFoundError, ValueError, OSError) as error:
        print(error)
    else:
        for figure in selected_figures:
            display(Image(filename=str(figure)))
